In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# Απενεργοποίηση του auto-scroll στην έξοδο του notebook
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def simulate_quantization(freq_s, num_bits):
    clear_output(wait=True)
    
    # 1. Calculations
    t_cont = np.linspace(0, 6e-3, 1000)
    x_cont = 0.6 * np.sin(2 * np.pi * 300 * t_cont) + 0.4 * np.cos(2 * np.pi * 700 * t_cont)
    x_cont = x_cont / np.max(np.abs(x_cont))

    T_s = 1.0 / freq_s
    t_samp = np.arange(0, 6e-3 + T_s/2, T_s)
    x_samp = 0.6 * np.sin(2 * np.pi * 300 * t_samp) + 0.4 * np.cos(2 * np.pi * 700 * t_samp)
    x_samp = x_samp / np.max(np.abs(x_samp))
    x_samp = np.clip(x_samp, -1.0, 1.0)

    num_levels = 2 ** num_bits
    Delta = 2.0 / num_levels
    quant_levels = -1.0 + (np.arange(num_levels) + 0.5) * Delta
    indices = np.clip(np.floor((x_samp + 1.0) / Delta), 0, num_levels - 1).astype(int)
    x_quantized = quant_levels[indices]
    error = x_quantized - x_samp

    # 2. Figure Setup
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1], width_ratios=[2, 1])
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    
    # Top Plot
    ax1.plot(t_cont*1e3, x_cont, 'b-', alpha=0.3, label='Continuous Signal')
    ax1.plot(t_samp*1e3, x_samp, 'ro', label='Samples')
    t_stairs = np.repeat(t_samp, 2)[1:] * 1e3
    x_stairs = np.repeat(x_quantized, 2)[:-1]
    ax1.plot(t_stairs, x_stairs, 'r--', drawstyle='steps-post', label='Quantized')
    ax1.plot(t_samp*1e3, x_quantized, 'g.', markersize=8, label='Levels')
    for i in range(len(t_samp)):
        ax1.plot([t_samp[i]*1e3, t_samp[i]*1e3], [x_samp[i], x_quantized[i]], 'k-', alpha=0.2)
    ax1.set_title('Signal Sampling & Quantization')
    ax1.set_ylabel('Amplitude [V]')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True)
    
    # Bottom Plot - Fixed Y-limits for visual comparison
    ax2.plot(t_samp*1e3, error, 'm.-')
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_ylim(-0.5, 0.5)
    ax2.set_title('Quantization Error e[n]')
    ax2.set_xlabel('Time [ms]')
    ax2.grid(True)

    # Right Side - Table
    ax3 = fig.add_subplot(gs[:, 1])
    ax3.axis('off')
    table_data = [[f"{x_samp[i]:.6f}", f"{x_quantized[i]:.6f}", f"{error[i]:.6f}"] for i in range(min(12, len(x_samp)))]
    table = ax3.table(cellText=table_data, colLabels=['Raw', 'Quantized', 'Error'], loc='center', cellLoc='center')
    table.scale(1, 1.5)
    
    ax3.text(0.5, 0.85, f"Parameters:\nDelta = {Delta:.6f}\nBits = {num_bits}\nLevels = {num_levels}", ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

# Layout management
display(Markdown("""
### User Guide
* **Fs (Hz):** Adjust the sampling frequency (number of samples per second).
* **Bits (k):** Adjust bit-depth to change the number of quantization levels ($2^k$).
* **Visuals:** Blue = Continuous, Red Dots = Samples, Green = Quantized Levels, Magenta = Error.
* **Observation:** As you increase 'Bits', notice the error signal in the bottom plot visually shrink.
"""))

freq_slider = widgets.IntSlider(value=2000, min=500, max=5000, step=500, description='Fs (Hz):')
bits_slider = widgets.IntSlider(value=3, min=2, max=16, step=1, description='Bits (k):')
ui = widgets.VBox([freq_slider, bits_slider])
display(ui)

out = widgets.interactive_output(simulate_quantization, {'freq_s': freq_slider, 'num_bits': bits_slider})
display(out)